# K-Means Clustering

This model clusters the *provinces* into k-groups to the nearest randomly generated centroid using Euclidean distance.

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from methods.read_data import createMatrix, printMatrix
from methods.normalize_data import normalizeData, standardizeData, coalesceData
from models.clustering import k_means_clustering, distance
from evaluation.conf_matrix import evaluate
from evaluation.holdout_validation import holdout_validation
from evaluation.kfold_cross_validation import kFoldSplit, crossValidation, accuracy
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from os import path
from copy import deepcopy
import csv
import random

### Step 1: Create the Dataset from the `PROJECT_DATASET` folder
- This will create a matrix with 12 fields/attributes and 87 provinces
- The 4 districts of NCR are used as its provinces, and Isabela City and Cotabato City are separate rows like in the poverty tables
- The internet use dataset is by region only, so the share of family spending on Information and Communication (FIES 2023 Table 7) is used instead

In [3]:
data_set_dir = path.join(path.abspath("."), "PROJECT_DATASET")          # Get the path to the dataset folder
provinces, fields, matrix = createMatrix(Path(data_set_dir))            # Create the matrix from the dataset
printMatrix(provinces, fields, matrix)                                  # Print the matrix with all of the provinces and fields

87 provinces x 12 fields
showing fields 0 to 11 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6 | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Table_7 | Information and Communication
  [9] DERIVED | Change in poverty incidence 2021 to 2023 (percentage points)
  [10] DERIVED | Change in poverty incidence 2018 to 2023 (percentage points)
  [11] DERIVED | Income inequality (top 10% share / bottom 40% share)

Province                       [0]         [1]         [2]         [3]         [4]         [5]         [6]         [7]         [8]         [9]        [10]        [11]
NCR 1st District               2.4         1.7         3.0         0.1       46.54     37711.0      482.79

### Step 2: Cleanup the data
- Convert the matrix into a dictionary of province statistics with province name as key and list of attributes as values
- Coalesce missing values to 0 for clustering, then standardize it using Z-score Scaling

In [4]:

# Coalesce NA values to 0 for clustering, then standardize it.
cleaned_data = standardizeData(coalesceData(matrix))

# Create dictionary of province statistics with province name as key and list of features as value
province_stats = {provinces[i]: cleaned_data[i] for i in range(len(provinces))}


### Step 3: Run the K-means clustering with k=5
- Run the k-means clustering with k set to 5
- Then display the categories for each province in the Dataset

In [5]:
copied_province_stats = deepcopy(province_stats)
categories = k_means_clustering(copied_province_stats, 9)
# For each labeled dataset, it includes the (1) label/cluster, (2) name of the province, and (3) data points associated with the province
print("\nLabeled Dataset:")
for id, points in province_stats.items():
  print(f"{categories[id]:<1} : {id:<22} {points}")

Initial Centroids
(1.0677106404454555, 0.16849348345858803, -0.10662278160007008, 0.9934330331978507, -0.6150591689312193, -0.32467219845580925, -0.38492143018665537, 0.0578264038575246, -0.6820962987496368, 1.514695728499007, 1.2043450247374232, 1.2014761438125203)
(-1.0073344429064803, -0.9270430643162131, -1.0708828646776265, -0.8709759566865594, 0.5263048385344948, 2.5302946694576502, 1.879994171891771, 1.5355126405931665, 1.6049943227002017, -0.03894737500856419, 0.43632958028495505, -0.6835623436001436)
(1.3889519294999388, 0.7162617573459885, 0.6133580804311721, 1.9256375281400555, 0.5359230745524645, -1.0774629584890107, -0.23464144610073592, -0.8137901690583523, -0.37715088255632473, 1.0707976989254153, 0.5571409985134332, 4.758726192639644)
(-1.4588086869830519, -1.523865213477112, -1.263734881293138, -1.137320098098618, -0.6665701662719004, 1.366446211263408, 2.893222105831889, 3.4814847675609677, 1.3152961773165555, 0.27178124569295004, 0.24648306592591804, -0.5923508038866

### Step 4: Evaluate the clusters using a Confusion Matrix
- Read the actual class of each province from the tagged dataset `province_policy_results.csv` (created by `methods/label_data.py` from the policies in the Policy for Labeling Data section of the project document)
- K-means only gives a cluster number, so each cluster is labeled with the most common actual class among its provinces (majority vote)
- Compare the predicted class with the actual class and compute the Recall, Precision, F1 Score and Accuracy for each class (one vs. rest)

In [6]:
# Read the actual class of each province from the tagged dataset
tagged_dir = path.join(data_set_dir, "province_policy_results.csv")    # Get the path to the tagged dataset
with open(tagged_dir, newline="", encoding="utf-8-sig") as file:
  actual_classes = {row["Province"]: row["Class"] for row in csv.DictReader(file)}

# Label each cluster with the most common actual class among its provinces (majority vote)
cluster_labels = {}
for cluster in set(categories.values()):
  cluster_classes = [actual_classes[id] for id in categories if categories[id] == cluster]
  cluster_labels[cluster] = max(cluster_classes, key=cluster_classes.count)

# Display the class assigned to each cluster
print("Cluster Labels:")
for cluster, label in sorted(cluster_labels.items()):
  print(f"{cluster:<1} : {label}")

# Create the list of actual and predicted classes for each province
actual = [actual_classes[id] for id in provinces]                     # Actual class from the tagged dataset
predicted = [cluster_labels[categories[id]] for id in provinces]      # Predicted class from the cluster label

print("\nActual vs. Predicted:")
for i in range(len(provinces)):
  print(f"{provinces[i]:<22} - {actual[i]:<9} {predicted[i]}")
print()

# Calculate and display the confusion matrix and the performance metrics for each class
results = evaluate(actual, predicted)

Cluster Labels:
0 : INCREASE
1 : DECREASE
2 : RETAIN
3 : DECREASE
4 : INCREASE
5 : RETAIN
6 : RETAIN
7 : INCREASE
8 : INCREASE

Actual vs. Predicted:
NCR 1st District       - DECREASE  DECREASE
NCR 2nd District       - DECREASE  DECREASE
NCR 3rd District       - DECREASE  RETAIN
NCR 4th District       - DECREASE  DECREASE
Abra                   - RETAIN    RETAIN
Apayao                 - RETAIN    RETAIN
Benguet                - RETAIN    RETAIN
Ifugao                 - RETAIN    RETAIN
Kalinga                - RETAIN    RETAIN
Mt. Province           - RETAIN    RETAIN
Ilocos Norte           - RETAIN    DECREASE
Ilocos Sur             - RETAIN    RETAIN
La Union               - RETAIN    RETAIN
Pangasinan             - RETAIN    INCREASE
Batanes                - RETAIN    DECREASE
Cagayan                - RETAIN    RETAIN
Isabela                - RETAIN    RETAIN
Nueva Vizcaya          - RETAIN    RETAIN
Quirino                - RETAIN    RETAIN
Aurora                 - RETAIN    RETAI

### Step 5: Wrap the K-means clustering as a model for the validation
- Holdout validation and k-fold cross validation both need a `model(testing_rows, training_list)` function, where the testing rows have no label
- K-means is *unsupervised*, so only the training rows are clustered, and each cluster is labeled with the most common actual class among its training rows (majority vote)
- A testing row gets the class of its nearest centroid, so the testing rows are never used to form the clusters or their labels
- The number of clusters is the same as in Step 3

In [7]:
clusters = 9                                                  # Same number of clusters as in Step 3

def kmeans_model(testing_rows, training_list):
  train_stats = {i: training_list[i][:-1] for i in range(len(training_list))}   # Features only, row position as id
  train_labels = {i: training_list[i][-1] for i in range(len(training_list))}   # Actual class of each training row

  random.seed(1)                                              # Same starting centroids every run
  with redirect_stdout(StringIO()):                           # Hide the centroids
    train_categories = k_means_clustering(train_stats, clusters)

  # Get the centroid and the label (majority vote) of each cluster
  centroids = []
  centroid_labels = []
  for cluster in sorted(set(train_categories.values())):
    members = [id for id in train_categories if train_categories[id] == cluster]
    centroids.append([sum(train_stats[id][dim] for id in members) / len(members) for dim in range(len(testing_rows[0]))])
    member_classes = [train_labels[id] for id in members]
    centroid_labels.append(max(member_classes, key=member_classes.count))

  # A testing row gets the class of its nearest centroid
  output = []
  for row in testing_rows:
    distances = [distance(row, centroid) for centroid in centroids]
    output.append(row + [centroid_labels[distances.index(min(distances))]])
  return output

featuresList = [list(province_stats[id]) for id in provinces]   # Features of each province, same order as provinces
labelList = [actual_classes[id] for id in provinces]            # Actual class of each province

### Step 6: Holdout Validation
- 80% of the provinces are used for training and the remaining 20% for testing
- Only the testing provinces are shown, `/` means the predicted class is the same as the actual class, `x` means it is not

In [8]:
# Each result is a tuple (province index, actual class, predicted class, fold)
holdout_results = holdout_validation(featuresList, labelList, 0.8, kmeans_model)

print(f"Training provinces: {len(provinces) - len(holdout_results)}, Testing provinces: {len(holdout_results)}\n")
print(f"{'Province':<22} {'Actual':<9} {'Predicted':<9}")
for result in holdout_results:
  mark = "/" if result[1] == result[2] else "x"
  print(f"{provinces[result[0]]:<22} {result[1]:<9} {result[2]:<9} {mark}")

correct = sum(result[1] == result[2] for result in holdout_results)
print(f"\nHoldout accuracy: {correct}/{len(holdout_results)} = {accuracy(holdout_results):.2%}")

Training provinces: 70, Testing provinces: 17

Province               Actual    Predicted
NCR 1st District       DECREASE  RETAIN    x
NCR 4th District       DECREASE  RETAIN    x
Kalinga                RETAIN    RETAIN    /
La Union               RETAIN    RETAIN    /
Cagayan                RETAIN    RETAIN    /
Nueva Vizcaya          RETAIN    RETAIN    /
Batangas               DECREASE  RETAIN    x
Occidental Mindoro     INCREASE  INCREASE  /
Negros Oriental        RETAIN    INCREASE  x
Siquijor               DECREASE  RETAIN    x
Northern Samar         INCREASE  INCREASE  /
Samar                  INCREASE  INCREASE  /
Zamboanga Sibugay      INCREASE  INCREASE  /
Bukidnon               INCREASE  INCREASE  /
Camiguin               RETAIN    RETAIN    /
Cotabato               INCREASE  INCREASE  /
Dinagat Islands        INCREASE  INCREASE  /

Holdout accuracy: 12/17 = 70.59%


### Step 7: K-Fold Cross Validation
- The provinces are split into 5 folds, each fold is the testing set once and the other folds are the training set
- Every province is predicted exactly once, so the accuracy uses all of the provinces

In [9]:
folds = 5
foldList = kFoldSplit(len(featuresList), folds)

# Each result is a tuple (province index, actual class, predicted class, fold)
kfold_results = crossValidation(featuresList, labelList, foldList, kmeans_model)

for f in range(folds):
  fold_results = [result for result in kfold_results if result[3] == f + 1]
  correct = sum(result[1] == result[2] for result in fold_results)
  print(f"Fold {f + 1}: {correct}/{len(fold_results)} = {accuracy(fold_results):.2%}")

correct = sum(result[1] == result[2] for result in kfold_results)
print(f"\nK-fold accuracy: {correct}/{len(kfold_results)} = {accuracy(kfold_results):.2%}")

Fold 1: 12/18 = 66.67%
Fold 2: 11/18 = 61.11%
Fold 3: 13/17 = 76.47%
Fold 4: 11/17 = 64.71%
Fold 5: 12/17 = 70.59%

K-fold accuracy: 59/87 = 67.82%


### Step 8: Confusion Matrix of the K-Fold Cross Validation
- The confusion matrix in Step 4 uses clusters that were formed and labeled with all of the provinces
- Here the predictions from the k-fold cross validation are used, so each province was predicted by clusters that did not include it

In [10]:
actual = [result[1] for result in kfold_results]              # Actual class from the tagged dataset
predicted = [result[2] for result in kfold_results]           # Predicted class from the nearest centroid

# Calculate and display the confusion matrix and the performance metrics for each class
kfold_metrics = evaluate(actual, predicted)

Confusion Matrix (Positive class: DECREASE)
                      Predicted    
                  Positive Negative
Actual    Positive       6        4
          Negative       5       72
Recall:    0.6000
Precision: 0.5455
F1 Score:  0.5714
Accuracy:  0.8966

Confusion Matrix (Positive class: INCREASE)
                      Predicted    
                  Positive Negative
Actual    Positive      37        9
          Negative      11       30
Recall:    0.8043
Precision: 0.7708
F1 Score:  0.7872
Accuracy:  0.7701

Confusion Matrix (Positive class: RETAIN)
                      Predicted    
                  Positive Negative
Actual    Positive      16       15
          Negative      12       44
Recall:    0.5161
Precision: 0.5714
F1 Score:  0.5424
Accuracy:  0.6897

